In [3]:
import logging
import pathlib
from argparse import ArgumentParser, RawTextHelpFormatter
from dataclasses import dataclass
from functools import partial
from typing import Callable

import torch
import torchaudio
import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/pytorch_audio/audio/examples/asr/emformer_rnnt/')
from common import MODEL_TYPE_LIBRISPEECH
from torchaudio.pipelines import EMFORMER_RNNT_BASE_LIBRISPEECH
from torchaudio.pipelines import RNNTBundle

sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from corpus.single_word import SingleWordDataset as Dataset
from src.text import load_text_encoder


In [4]:
@dataclass
class Config:
    dataset: Callable
    bundle: RNNTBundle

In [5]:

_CONFIGS = {
    MODEL_TYPE_LIBRISPEECH: Config(
        partial(torchaudio.datasets.LIBRISPEECH, url="test-clean"),
        EMFORMER_RNNT_BASE_LIBRISPEECH)
}

In [23]:
dataset_path = '/om/data/public/librispeech/librispeech_data/librispeech_data/test-clean/'
model_type = 'librispeech'

In [24]:
dataset = _CONFIGS[model_type].dataset(dataset_path)
bundle = _CONFIGS[model_type].bundle
decoder = bundle.get_decoder().cuda()
token_processor = bundle.get_token_processor()
feature_extractor = bundle.get_feature_extractor().cuda()
streaming_feature_extractor = bundle.get_streaming_feature_extractor().cuda()
hop_length = bundle.hop_length
num_samples_segment = bundle.segment_length * hop_length
num_samples_segment_right_context = num_samples_segment + bundle.right_context_length * hop_length

In [34]:
for idx in range(10):
    sample = dataset[idx]
    waveform = sample[0].squeeze().cuda()
    # Streaming decode.
    state, hypothesis = None, None
    for idx in range(0, len(waveform), num_samples_segment):
        segment = waveform[idx : idx + num_samples_segment_right_context]
        segment = torch.nn.functional.pad(segment, (0, num_samples_segment_right_context - len(segment)))
        with torch.no_grad():
            features, length = streaming_feature_extractor(segment)
            hypos, state = decoder.infer(features, length, 10, state=state, hypothesis=hypothesis)
        hypothesis = hypos[0]
        transcript = token_processor(hypothesis.tokens, lstrip=False)
        print(transcript, end="", flush=True)
    print()

    # Non-streaming decode.
    with torch.no_grad():
        features, length = feature_extractor(waveform)
        hypos = decoder(features, length, 10)
    print(f'non-streaming: {token_processor(hypos[0].tokens)}')
    print(f"truth: {sample[2].lower()}")
    print()

 he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to beled out in thickard fat and sauce
non-streaming: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fat and sauce
truth: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce

 stuff it you his belly counselled him
non-streaming: stuff it into you his belly counselled him
truth: stuff it into you his belly counselled him

 after early nightfall the yellow lamps would light up here and there the squal of thephals
non-streaming: after early nightfall the yellow lamps would light up here and there the squalid quarter of the brothels
truth: after early nightfall the yellow lamps would light up here and there the squalid quarter of the brothels

 hello bertie any good in your mind
non-stre

In [29]:
sample[2]

'STUFF IT INTO YOU HIS BELLY COUNSELLED HIM'